Importing libraries to work on the dataset

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

Importing the dataset

In [3]:
df= pd.read_csv("data\creditcard.csv")

<>:1: SyntaxWarning: invalid escape sequence '\c'
<>:1: SyntaxWarning: invalid escape sequence '\c'
C:\Users\Shreyansh Rai\AppData\Local\Temp\ipykernel_7792\1077104441.py:1: SyntaxWarning: invalid escape sequence '\c'
  df= pd.read_csv("data\creditcard.csv")
C:\Users\Shreyansh Rai\AppData\Local\Temp\ipykernel_7792\1077104441.py:1: SyntaxWarning: invalid escape sequence '\c'
  df= pd.read_csv("data\creditcard.csv")


FileNotFoundError: [Errno 2] No such file or directory: 'data\\creditcard.csv'

In [ ]:
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [ ]:
df.shape

(284807, 31)

In [ ]:
df.columns.tolist()

['Time',
 'V1',
 'V2',
 'V3',
 'V4',
 'V5',
 'V6',
 'V7',
 'V8',
 'V9',
 'V10',
 'V11',
 'V12',
 'V13',
 'V14',
 'V15',
 'V16',
 'V17',
 'V18',
 'V19',
 'V20',
 'V21',
 'V22',
 'V23',
 'V24',
 'V25',
 'V26',
 'V27',
 'V28',
 'Amount',
 'Class']

Checkig and printing the missing values

In [ ]:
print("Missing values per column:\n", df.isnull().sum().loc[lambda s: s>0])

Missing values per column:
 Series([], dtype: int64)


Checking and printing the Duplicate Values

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 1081


Checking Traget values for fraud and legitmate values

In [ ]:
print("Class distribution:\n", df['Class'].value_counts())

Class distribution:
 Class
0    284315
1       492
Name: count, dtype: int64


Deleting the duplicate values

In [ ]:
if df.duplicated().sum() > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Duplicates dropped. New shape:", df.shape)

Duplicates dropped. New shape: (283726, 31)


Reducing the datatypes for memory reducing and effective working and training on the dataset

In [ ]:
for col in df.select_dtypes(include=['float64']).columns:
    df[col] = pd.to_numeric(df[col], downcast='float')

for col in df.select_dtypes(include=['int64']).columns:
    df[col] = pd.to_numeric(df[col], downcast='integer')

print(df.dtypes.head(10))


Time    float32
V1      float32
V2      float32
V3      float32
V4      float32
V5      float32
V6      float32
V7      float32
V8      float32
V9      float32
dtype: object


Extracting time, day of week, hour etc for further data values

In [ ]:
base = pd.to_datetime("2018-01-01")  # arbitrary base date
df['datetime'] = base + pd.to_timedelta(df['Time'], unit='s')

df['hour'] = df['datetime'].dt.hour.astype('int8')
df['dayofweek'] = df['datetime'].dt.dayofweek.astype('int8')
df['is_weekend'] = df['dayofweek'].isin([5,6]).astype('int8')


In [ ]:
df.tail()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V25,V26,V27,V28,Amount,Class,datetime,hour,dayofweek,is_weekend
283721,172786.0,-11.881118,10.071785,-9.834784,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,1.436807,0.250034,0.943651,0.823731,0.77,0,2018-01-02 23:59:46,23,1,0
283722,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,-0.606624,-0.395255,0.068472,-0.053527,24.79,0,2018-01-02 23:59:47,23,1,0
283723,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.265745,-0.087371,0.004455,-0.026561,67.88,0,2018-01-02 23:59:48,23,1,0
283724,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,-0.569159,0.546668,0.108821,0.104533,10.00,0,2018-01-02 23:59:48,23,1,0
283725,172792.0,-0.533413,-0.189733,0.703337,-0.506271,-0.012546,-0.649617,1.577006,-0.414650,0.486180,...,-0.473649,-0.818267,-0.002415,0.013649,217.00,0,2018-01-02 23:59:52,23,1,0


Scaling the data and compressing the large values

In [ ]:
df['log_amount'] = np.log1p(df['Amount'])
scaler = StandardScaler()
df['amount_scaled'] = scaler.fit_transform(df[['log_amount']])

joblib.dump(scaler, "creditcard_amount_scaler.joblib")

['creditcard_amount_scaler.joblib']

Creating matching values of both the dataset

In [ ]:
df = df.rename(columns={'Class': 'isFraud'})  

v_cols = [f'V{i}' for i in range(1,29)]
features = v_cols + ['amount_scaled', 'hour', 'dayofweek', 'is_weekend']

cleaned_cc = df[features + ['isFraud']].copy()

print("Cleaned dataset shape:", cleaned_cc.shape)
print("Columns:", cleaned_cc.columns.tolist())


Cleaned dataset shape: (283726, 33)
Columns: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'amount_scaled', 'hour', 'dayofweek', 'is_weekend', 'isFraud']


Saving the updated dataset

In [ ]:
cleaned_cc.to_csv("cleaned_creditcard.csv", index=False)
print("Saved cleaned_creditcard.csv and cleaned_creditcard.csv")


Saved cleaned_creditcard.csv and cleaned_creditcard.csv


In [ ]:
df.head()

NameError: name 'df' is not defined